# Hợp nhất Mô hình (Model Merge: Base Model + 1-LoRA Adapter)

Thực hiện việc **hợp nhất (merge) adapter LoRA** đã được huấn luyện ở Notebook 8 quay trở lại **mô hình nền Llama-3.1-8B gốc**.

**Tại sao cần Model Merge?**
- Sau khi hợp nhất, có một mô hình độc lập (single weights file) có thể được chạy trực tiếp qua HuggingFace, vLLM, hoặc chuyển đổi sang định dạng GGUF để chạy cục bộ bằng Ollama/Llama.cpp.
- Hợp nhất giúp tối ưu hóa đáng kể tốc độ suy luận (Inference Latency) so với việc nạp song song mô hình nền và adapter LoRA.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from dotenv import load_dotenv

# 1. Nạp các biến môi trường cấu hình cache trước khi load bất kỳ thư viện nào khác
load_dotenv(os.path.abspath("../.env"))

# 2. --- BẢN VÁ LỖI UNPICKLING CHO PYTORCH ---
# Phải chạy trước khi load Unsloth để tránh lỗi PyTorch Security khi nạp dữ liệu tuần tự hóa
import torch
import numpy as np
try:
    from numpy.core.multiarray import _reconstruct
except ImportError:
    from numpy._core.multiarray import _reconstruct

try:
    from numpy import dtype, ndarray
except ImportError:
    dtype = np.dtype
    ndarray = np.ndarray

safe_globals = [_reconstruct, dtype, ndarray, np.float32, np.float64, np.int64]
try:
    import numpy.dtypes
    for name in dir(numpy.dtypes):
        attr = getattr(numpy.dtypes, name)
        if isinstance(attr, type):
            safe_globals.append(attr)
except (ImportError, AttributeError):
    pass
torch.serialization.add_safe_globals(safe_globals)

# 3. IMPORT UNSLOTH TRƯỚC TRANSFORMERS để đảm bảo tối ưu hóa và tránh lỗi cảnh báo
from unsloth import FastLanguageModel

# 4. Import Transformers sau Unsloth và áp dụng bản vá bảo mật
import transformers
import transformers.utils.import_utils
import transformers.utils
import transformers.trainer
import transformers.modeling_utils
import transformers.trainer_utils
for module in [
    transformers.utils.import_utils,
    transformers.utils,
    transformers.trainer,
    transformers.modeling_utils,
    transformers.trainer_utils
]:
    if hasattr(module, "check_torch_load_is_safe"):
        module.check_torch_load_is_safe = lambda: None

# 5. --- BẢN VÁ LỖI MERGED_4BIT CHO TRANSFORMERS ---
# Tránh lỗi NotImplementedError khi save_pretrained_merged bản 4-bit ở phiên bản transformers mới
try:
    import transformers.core_model_loading
    transformers.core_model_loading.revert_weight_conversion = lambda model, state_dict: state_dict
    print("✔ Đã áp dụng bản vá revert_weight_conversion cho core_model_loading")
except (ImportError, AttributeError):
    pass

try:
    import transformers.modeling_utils
    transformers.modeling_utils.revert_weight_conversion = lambda model, state_dict: state_dict
    print("✔ Đã áp dụng bản vá revert_weight_conversion cho modeling_utils")
except (ImportError, AttributeError):
    pass
# ----------------------------------------

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
✔ Đã áp dụng bản vá revert_weight_conversion cho core_model_loading
✔ Đã áp dụng bản vá revert_weight_conversion cho modeling_utils


## 1. Thiết lập Cấu hình & Tải Adapter và Mô hình nền

In [2]:
max_seq_length = 1024 # Điều chỉnh về 1024 để khớp với cấu hình huấn luyện ở Notebook 8 (tiết kiệm VRAM)
dtype = None
load_in_4bit = True # Giữ nguyên chế độ 4-bit để tiết kiệm RAM trên laptop

ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"

print(f"Đang tải mô hình nền và adapter LoRA từ: {ADAPTER_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Đang tải mô hình nền và adapter LoRA từ: ../adapters/llama_8b_1lora_aes...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 58.68it/s]
Unsloth: Will load ../adapters/llama_8b_1lora_aes as a legacy tokenizer.
Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 2. Thực hiện Hợp nhất và Lưu trữ Cục bộ

Unsloth hỗ trợ 2 chế độ hợp nhất chính:
1. **`merged_16bit`**: Xuất ra mô hình đầy đủ độ chính xác float16 (~16 GB). Đây là phương án khuyến nghị để export sang GGUF hoặc vLLM triển khai ở môi trường sản xuất.
2. **`merged_4bit`**: Xuất ra mô hình đã được lượng hóa 4-bit sẵn (~5.5 GB). Tiết kiệm ổ cứng tối đa.

In [3]:
MERGED_MODEL_DIR_16BIT = "../merged_model_16bit"
MERGED_MODEL_DIR_4BIT = "../merged_model_4bit"

# 1. Xuất mô hình Merged 16-bit
try:
    print(f"Đang hợp nhất và lưu mô hình Float16 tại: {MERGED_MODEL_DIR_16BIT}...")
    model.save_pretrained_merged(
        MERGED_MODEL_DIR_16BIT,
        tokenizer,
        save_method = "merged_16bit",
        maximum_memory_usage = 0.4,
    )
    print("✔ Hoàn thành lưu mô hình 16-bit!")
except Exception as e:
    print(f"❌ Lỗi lưu bản 16-bit do thiếu RAM/VRAM hệ thống: {e}")
    print("Bỏ qua và chuyển sang lưu mô hình lượng hóa 4-bit...")

# 2. Xuất mô hình Merged 4-bit (Nhẹ hơn, tiết kiệm ổ cứng)
print(f"\nĐang hợp nhất và lưu mô hình lượng hóa 4-bit tại: {MERGED_MODEL_DIR_4BIT}...")
model.save_pretrained_merged(
    MERGED_MODEL_DIR_4BIT,
    tokenizer,
    save_method = "merged_4bit_forced",
    maximum_memory_usage = 0.4,
)
print("✔ Hoàn thành lưu mô hình 4-bit!")

Đang hợp nhất và lưu mô hình Float16 tại: ../merged_model_16bit...


Unsloth: Restored added_tokens_decoder metadata in ../merged_model_16bit\tokenizer_config.json.


Found HuggingFace hub cache directory: T:\5 - Summer 2026\AES_LLM\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Merge process complete. Saved to `t:\5 - Summer 2026\AES_LLM\merged_model_16bit`
✔ Hoàn thành lưu mô hình 16-bit!

Đang hợp nhất và lưu mô hình lượng hóa 4-bit tại: ../merged_model_4bit...


Unsloth: Restored added_tokens_decoder metadata in ../merged_model_4bit\tokenizer_config.json.
t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\peft\tuners\lora\bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Unsloth: Merging LoRA weights into 4bit model...
Unsloth: Merging finished.
Unsloth: Found skipped modules: ['lm_head']. Updating config.
Unsloth: Saving merged 4bit model to ../merged_model_4bit...


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.14s/it]


Unsloth: Merged 4bit model saved.
Unsloth: Merged 4bit model process completed.
✔ Hoàn thành lưu mô hình 4-bit!


## 3. Hướng dẫn Đẩy mô hình lên Hugging Face Hub (Tùy chọn)

Nếu muốn sao lưu hoặc chia sẻ với cộng đồng, bạn có thể đẩy thẳng mô hình đã merge lên Hugging Face Hub:

In [4]:
# # Đăng nhập Hugging Face bằng token ghi (write token)
# # huggingface-cli login

# # Lệnh đẩy lên Hub:
# model.push_to_hub_merged(
#     "username/llama-3.1-8b-aes-1lora", 
#     tokenizer, 
#     save_method = "merged_16bit", 
#     token = "your_hf_token"
# )

## 4. Xuất Mô hình sang Định dạng GGUF (Lượng hóa Q4_K_M)

Tự động lượng hóa mô hình đã nạp và lưu thành định dạng GGUF phục vụ cho việc suy luận cục bộ bằng Ollama/Llama.cpp.

In [ ]:
# 1. Nạp cấu hình môi trường và import các thư viện cần thiết (chạy riêng biệt)
if 'model' not in globals() or 'tokenizer' not in globals():
    import os
    from dotenv import load_dotenv
    load_dotenv(os.path.abspath("../.env"))
    
    from unsloth import FastLanguageModel
    
    # Chỉ định đường dẫn tới mô hình đã hợp nhất Float16
    model_path = "../merged_model_16bit"
    if not os.path.exists(model_path):
        # Fallback về adapter nếu chưa có merged model
        model_path = "../adapters/llama_8b_1lora_aes"
        
    print(f"Đang nạp mô hình từ: {model_path}...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_path,
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = False,
    )
else:
    print("Mô hình đã được nạp sẵn trong bộ nhớ, sử dụng trực tiếp...")

# 2. Xuất mô hình sang định dạng GGUF lượng hóa Q4_K_M
print("Đang lượng hóa và xuất mô hình sang tệp model-q4_k_m.gguf...")
model.save_pretrained_gguf(
    "../merged_model_4bit_from_16bit", 
    tokenizer, 
    quantization_method = "q4_k_m"
)
print("✔ Hoàn thành xuất tệp GGUF tại: ../merged_model_4bit_from_16bit/model-q4_k_m.gguf")